# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [ ]:
import os
import sqlite3
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from datetime import datetime

In [ ]:
load_dotenv()
# If not using .env, manually set it here: 
# os.environ["OPENROUTER_API_KEY"] = "sk-or-v1-your-key-here"

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

In [ ]:
def init_db():
    conn = sqlite3.connect("research_agent.db")
    cursor = conn.cursor()
    cursor.execute('''CREATE TABLE IF NOT EXISTS logs 
                      (timestamp TEXT, model TEXT, prompt TEXT, response TEXT)''')
    conn.commit()
    conn.close()

def log_interaction(model, prompt, response):
    conn = sqlite3.connect("research_agent.db")
    cursor = conn.cursor()
    cursor.execute("INSERT INTO logs VALUES (?, ?, ?, ?)", 
                   (datetime.now().strftime("%Y-%m-%d %H:%M:%S"), model, prompt, response))
    conn.commit()
    conn.close()

init_db()
print("✅ Database and Client Initialized.")

In [ ]:
import json

# Simulating a technical database search
def search_tech_docs(query):
    """Searches the internal technical manual for documentation."""
    docs = {
        "api": "API stands for Application Programming Interface. It allows software to talk.",
        "gradio": "Gradio is an open-source Python library used to build ML web apps.",
        "sqlite": "SQLite is a C-language library that implements a small, fast SQL database engine."
    }
    # Return the doc if keyword found, else a default message
    for key in docs:
        if key in query.lower():
            return docs[key]
    return "No specific documentation found for this topic."

# Define the tool for the OpenAI/OpenRouter format
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_tech_docs",
            "description": "Search the internal technical manual for coding definitions.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The technical topic to look up."}
                },
                "required": ["query"],
            },
        },
    }
]

In [ ]:
import json

# Simulating a technical database search
def search_tech_docs(query):
    """Searches the internal technical manual for documentation."""
    docs = {
        "api": "API stands for Application Programming Interface. It allows software to talk.",
        "gradio": "Gradio is an open-source Python library used to build ML web apps.",
        "sqlite": "SQLite is a C-language library that implements a small, fast SQL database engine."
    }
    # Return the doc if keyword found, else a default message
    for key in docs:
        if key in query.lower():
            return docs[key]
    return "No specific documentation found for this topic."

# Define the tool for the OpenAI/OpenRouter format
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_tech_docs",
            "description": "Search the internal technical manual for coding definitions.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The technical topic to look up."}
                },
                "required": ["query"],
            },
        },
    }
]

In [ ]:
def agent_chat(message, history, model_name, system_prompt):
    models = {
        "Claude 3.5 Sonnet": "anthropic/claude-3.5-sonnet",
        "GPT-4o": "openai/gpt-4o",
        "Gemini 1.5 Pro": "google/gemini-pro-1.5"
    }
    
    messages = [{"role": "system", "content": system_prompt}]
    for user_msg, assistant_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})
    messages.append({"role": "user", "content": message})

    # Step 1: Initial Call to see if Tool is needed
    response = client.chat.completions.create(
        model=models[model_name],
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )
    
    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls

    # Step 2: Handle Tool Call if it exists
    if tool_calls:
        for tool_call in tool_calls:
            function_args = json.loads(tool_call.function.arguments)
            observation = search_tech_docs(function_args.get("query"))
            
            messages.append(response_message)
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": "search_tech_docs",
                "content": observation,
            })
    
    # Step 3: Stream the final response
    final_stream = client.chat.completions.create(
        model=models[model_name],
        messages=messages,
        stream=True
    )

    full_text = ""
    for chunk in final_stream:
        if chunk.choices[0].delta.content:
            full_text += chunk.choices[0].delta.content
            yield full_text
            
    # Step 4: Log to SQLite
    log_interaction(model_name, message, full_text)

In [ ]:
import pandas as pd
import sqlite3
import gradio as gr

# Function to pull data from SQLite for the Gradio UI
def get_history():
    try:
        conn = sqlite3.connect("research_agent.db")
        df = pd.read_sql_query("SELECT * FROM logs ORDER BY timestamp DESC", conn)
        conn.close()
        return df
    except Exception as e:
        return pd.DataFrame({"Status": ["No history found yet. Start a chat!"]})

# --- Gradio UI Layout (Day 5: Blocks & Tabs) ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("#  Research & Logging Agent")
    gr.Markdown("### Week 2 Exercise")
    
    with gr.Tabs():
        # TAB 1: AI Researcher
        with gr.TabItem("AI Researcher"):
            with gr.Row():
                model_sel = gr.Dropdown(
                    choices=["Claude 3.5 Sonnet", "GPT-4o", "Gemini 1.5 Pro"], 
                    value="Claude 3.5 Sonnet", 
                    label="Agent Brain"
                )
                sys_prompt = gr.Textbox(
                    value="You are a Senior Technical Researcher. Use your tools to provide accurate data.", 
                    label="System Instructions"
                )
            
            # Streaming Chat Interface
            gr.ChatInterface(
                fn=agent_chat, 
                additional_inputs=[model_sel, sys_prompt], 
                type="tuples"
            )
            
        # TAB 2: Database Logs (Fixed Indentation)
        with gr.TabItem("Database Logs"):
            gr.Markdown("### 📜 Conversation History")
            gr.Markdown("Data below is pulled in real-time from `research_agent.db`.")
            
            refresh_btn = gr.Button("🔄 Refresh History", variant="primary")
            
            # Displays the SQLite data as a clean table
            history_table = gr.DataFrame(value=get_history)
            
            # Link the button to the refresh function
            refresh_btn.click(fn=get_history, outputs=history_table)

# Launch the app
demo.launch(share=True, debug=True)